# Synthesis record curation

Normalize positive and enumerated negative extraction tables, inspect stage reports, and prepare the inputs to dataset construction. Chemical rules are implemented in `mofinder.curation`.

## Inputs and setup

Install `python -m pip install -e ".[curation,notebook]"` from the repository root. Run the cells in order after extraction. Curation uses saved tables and requires no API key or PDF access. All paths below are repository-relative.

| Input | Default location | Preparation |
| --- | --- | --- |
| Positive extraction, CSV | `results/extraction/positive/mof_extraction.csv` | Output of notebook 03, with the extraction schema retained. |
| Enumerated negatives, CSV | `results/extraction/negative/mof_extraction_failures_enum.csv` | Output of notebook 04 after enumeration. |
| Linker molecular weights, CSV | `data/lookups/linker_molecular_weights.csv` | Included, with two headerless columns: linker name and molecular weight. |
| Linker prime corrections, JSON | `data/lookups/linker_prime_corrections.json` | Included exact DOI/name pairs for documented prime glyph restorations. |

Select different CSV paths in `configs/curation.json` when using another extraction run. For an immediate test with supplied raw records, open [the cleaning demo](../Demo/01_data_cleaning/demo.ipynb); its document-availability flags replace local PDF paths. Archived stage 6 tables are already cleaned and belong in dataset preparation.

Leave `RUN_CURATION = False` to inspect inputs only. When enabled, numbered tables and reports are saved under `results/curation/positive/` and `results/curation/negative/`. Run notebook 06 after both branches reach stage 6.

Implementation: [stage order](../src/mofinder/curation/pipeline.py), [curation modules](../src/mofinder/curation/), and [duration parsing](../src/mofinder/curation/times.py). See the [source-to-code guide](../docs/source_to_code.md) for the original workflow stages and their corresponding functions.


In [ ]:
from pathlib import Path
import json

from mofinder.display import display_paths
from mofinder.curation import load_config, validate_inputs, run_pipeline

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "configs" / "curation.json").is_file()
)
CONFIG_PATH = PROJECT_ROOT / "configs" / "curation.json"
settings = load_config(CONFIG_PATH)


## Inputs

Set the extraction paths in `configs/curation.json`. The configuration selects the headerless molecular-weight lookup at `data/lookups/linker_molecular_weights.csv`. Validation reports known and unresolved weights without creating or modifying files.


In [ ]:
validation = validate_inputs(settings, mode="both")
print(json.dumps(display_paths(validation), indent=2))


## Run curation

Each branch writes numbered intermediate CSVs and reports. Positive stage 7 applies the optional DOI-level trimming; dataset preparation defaults to stage 6. Set `TRIM_POSITIVE` to `False` to finish at stage 6.


In [ ]:
RUN_CURATION = False
TRIM_POSITIVE = True

if RUN_CURATION:
    results = run_pipeline(
        settings, mode="both", reports=True, plots=False,
        trim_positive=TRIM_POSITIVE,
    )
    print(json.dumps(display_paths(results), indent=2))


## Reports and chemical checks

Stage reports and metal-linker coverage tables are saved under `results/curation/<branch>/reports/`. Each completed branch also records input and lookup hashes in `curation_manifest.json`.

Positive and negative branches use different rules for temperatures, pore outliers, linker aliases, and synthesis descriptions. Both branches map `h3btb` to `1,3,5-Tris(4-carboxyphenyl)benzene`. Molar-mass calculations account for complete hydrate groups and bracketed complexes. See [curation documentation](../docs/curation.md) before final dataset generation.
